In [1]:
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    
)
from datasets import Dataset
import mlflow
import os
import torch_directml
import pandas as pd


u:\CODE\ReelSense\etl\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\catas\AppData\Local\Temp\ipykernel_13700\1851196425.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (


In [2]:
os.environ["HF_HOME"]= "U:\\CODE\\models"
mlflow.set_experiment("ReelSense-Experiment")

device = torch_directml.device()

In [3]:
from transformers import TrainerCallback

class EnhancedMLflowCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and mlflow.active_run():
            for key, value in logs.items():
                if isinstance(value, (int, float)):
                    mlflow.log_metric(key, value, step=state.global_step)

In [4]:
with mlflow.start_run(run_name="ReelSenseMovie_Embeddings_FineTuning"):
    mlflow.log_param("model_name", "all-MiniLM-L6-v2")
    mlflow.log_param("dataset", "Letterboxd_Movies")
    mlflow.log_param("sample_size", 20000)

    print("Loading dataset...")

    movies = pd.read_csv("../data/movies_enriched.csv").head(20000)

    columnas = [
        "name",
        "description",
        "tagline",
        "main_actors",
        "main_directors",
        "main_genres",
        "main_themes",
    ]
    movies[columnas] = movies[columnas].fillna("")

    movies["anchor"] = movies[["name", "main_actors", "main_directors", "main_genres", "main_themes"]].agg(" ".join, axis=1)
    movies["positive"] = movies[["description", "tagline"]].agg(" ".join, axis=1)

    # ✅ SentenceTransformerTrainer espera un Dataset de HuggingFace
    train_dataset = Dataset.from_dict({
        "anchor": movies["anchor"].tolist(),
        "positive": movies["positive"].tolist(),
    })
    model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

    train_loss = losses.MultipleNegativesRankingLoss(model)
    
    training_args = SentenceTransformerTrainingArguments(
        output_dir="./models/reelsense-movie-embeddings2",
        num_train_epochs=1,
        per_device_train_batch_size=32,
        warmup_steps=100,
        logging_steps=10,        # ✅ Cada 10 steps dispara on_log → MLflow
        save_strategy="no",
        report_to="none",        # Desactivamos el reporte built-in, usamos el nuestro
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        loss=train_loss,
        callbacks=[EnhancedMLflowCallback()],  # ✅ Ahora sí se engancha
    )


    print("Training in Radeon 7600")
    trainer.train()


    model_path = "./models/reelsense-movie-embeddings"
    model.save(model_path)
    mlflow.sentence_transformers.log_model(
        model=model,
        artifact_path="model_output",
        input_example=["Sample text for inference"],
    )
    print("Model fine-tuning completed and saved to /models/reelsense-movie-embeddings")

Loading dataset...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4017.68it/s]


Training in Radeon 7600


Step,Training Loss
10,1.881308
20,1.738161
30,1.592381
40,1.365494
50,1.303358
60,1.183700
70,1.120802
80,1.019877
90,0.998892
100,0.964937


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.29it/s]
2026/05/05 14:39:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 14:39:32 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: ValueError('Unsupported input type: Series. Expected one of: str, dict, PIL.Image.Image, np.ndarray, torch.Tensor'). To see the full traceback, set the logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)`.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8232.75it/s]


Model fine-tuning completed and saved to /models/reelsense-movie-embeddings


In [ ]:
#from transformers import AutoTokenizer

#enriched_movies = pd.read_csv("../data/movies_enriched.csv")

# Cargar la herramienta que cuenta los tokens
#tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Sacar el primer "Super Texto"
#texto_prueba = enriched_movies[['name', 'description', 'main_actors', 'main_directors', 'main_genres', 'main_themes']].iloc[0]
#texto_prueba = ' '.join(texto_prueba.astype(str))
#tokens = tokenizer.encode(texto_prueba)

#print(f"Caracteres: {len(texto_prueba)}")
#print(f"Tokens reales: {len(tokens)}")

: 

In [5]:
#mflow stop experiment run
mlflow.end_run()